# 08-03 깊은 CNN으로 MNSIT 분류하기

## 모델 이해
* 1번 레이어 : 합성곱층  
합성곱(입력채널=1, 출력채널=32, 커널사이즈=3, 스트라이드=1, 패딩=1) + ReLU  
맥스풀링(커널사이즈=2, 스트라이드=2)  

* 2번 레이어 : 합성곱층  
합성곱(입력채널=32, 출력채널=64, 커널사이즈=3, 스트라이드=1, 패딩=1) + ReLU  
맥스풀링(커널사이즈=2, 스트라이드=2)  

* 3번 레이어 : 합성곱층  
합성곱(입력채널=64, 출력채널=128, 커널사이즈=3, 스트라이드=1, 패딩=1) + ReLU  
맥스풀링(커널사이즈=2, 스트라이드=2, 패딩=1)  

* 4번 레이어 : 전결합층  
특성맵을 펼친다.  
전결합층(뉴런 625개) + ReLU  

* 5번 레이어 : 전결합층  
전결합층(뉴런 10개) + Softmax

In [1]:
import torch
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torch.nn.init

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
torch.manual_seed(777)
if device == 'cuda':
    torch.cuda.manual_seed_all(777)

cuda


In [3]:
learning_rate = 0.001
training_epochs = 15
batch_size = 100

In [4]:
mnist_train = datasets.MNIST(root='MNIST_data/',
                             train=True,
                             transform=transforms.ToTensor(),
                             download=True)
mnist_test = datasets.MNIST(root='MNIST_data/',
                            train=False,
                            transform=transforms.ToTensor(),
                            download=True)
data_loader = torch.utils.data.DataLoader(dataset=mnist_train,
                                          batch_size=batch_size,
                                          shuffle=True,
                                          drop_last=True)

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 466kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.45MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 10.5MB/s]


In [5]:
class CNN(torch.nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.layer1 = torch.nn.Sequential(
            torch.nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2))
        self.layer2 = torch.nn.Sequential(
            torch.nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2))
        self.layer3 = torch.nn.Sequential(
            torch.nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2, padding=1))
        #128채널 , 4*4 크기의 이미지(28x28 -> 14x14 -> 7x7 -> 4x4)
        self.fc1 = torch.nn.Linear(4 * 4 * 128, 625, bias=True)
        self.fc2 = torch.nn.Linear(625, 10)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = self.layer3(out)
        out = out.view(out.size(0), -1)
        out = self.fc1(out)
        out = torch.nn.functional.relu(out)
        out = self.fc2(out)
        return out

In [6]:
model = CNN().to(device)
criterion = torch.nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

total_batch = len(data_loader)
print('총 배치의 수 : {}'.format(total_batch))

총 배치의 수 : 600


In [7]:
for epoch in range(training_epochs):
    avg_cost = 0

    for X, Y in data_loader:
        X = X.to(device)
        Y = Y.to(device)

        optimizer.zero_grad()
        hypothesis = model(X)
        cost = criterion(hypothesis, Y)
        cost.backward()
        optimizer.step()

        avg_cost += cost / total_batch

    print('Epoch: {:04d}, Cost: {:.9f}'.format(epoch + 1, avg_cost))

Epoch: 0001, Cost: 0.179614604
Epoch: 0002, Cost: 0.044623666
Epoch: 0003, Cost: 0.030700943
Epoch: 0004, Cost: 0.022829186
Epoch: 0005, Cost: 0.018908143
Epoch: 0006, Cost: 0.014119077
Epoch: 0007, Cost: 0.013703456
Epoch: 0008, Cost: 0.011207722
Epoch: 0009, Cost: 0.009969781
Epoch: 0010, Cost: 0.008692060
Epoch: 0011, Cost: 0.007885974
Epoch: 0012, Cost: 0.005781307
Epoch: 0013, Cost: 0.008154275
Epoch: 0014, Cost: 0.004716746
Epoch: 0015, Cost: 0.005335110


In [8]:
with torch.no_grad():
    X_test = mnist_test.test_data.view(len(mnist_test), 1, 28, 28).float().to(device)
    Y_test = mnist_test.test_labels.to(device)

    prediction = model(X_test)
    correct_prediction = torch.argmax(prediction, 1) == Y_test
    accuracy = correct_prediction.float().mean()
    print('Accuracy: {:.4f}'.format(accuracy.item()))

Accuracy: 0.9910


/usr/local/lib/python3.12/dist-packages/torchvision/datasets/mnist.py:81: UserWarning: test_data has been renamed data
  warnings.warn("test_data has been renamed data")
/usr/local/lib/python3.12/dist-packages/torchvision/datasets/mnist.py:71: UserWarning: test_labels has been renamed targets
  warnings.warn("test_labels has been renamed targets")


층을 더 깊게 쌓았는데, 정확도는 오히려 줄었어야 했는데... 안줄었네 ㅋㅋ 그래도 필요한만큼 쌓는게 중요하다.